# V12 — Multi-Seed Experiment

## Purpose
Run run experiment with seeds 0, 1, 2 (run used seed 42) to get variance estimates.

## Setup
- Add dataset `rojanregmi1/cifar100-c` via Add Data
- GPU T4 x2 + Internet ON
- Focus: CIFAR-100-C, severity 5, lr=0.005 and lr=0.05 (safe + aggressive)

## What we measure
- Accuracy mean ± std across 4 seeds (0, 1, 2, 42)
- Collapse rate consistency across seeds
- Seed 42 results must match run (sanity check)

**Time:** ~90 min (3 seeds × 30 min each, 2 LRs instead of 4)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import time
import json
import copy
import gc
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

NUM_CLASSES = 100
BATCH_SIZE = 64
N_SAMPLES = 5000
CORRUPTIONS = ['gaussian_noise', 'defocus_blur', 'snow', 'contrast', 'jpeg_compression']
SEVERITIES = [5]
LR_LIST = [0.005, 0.05]
SEEDS = [0, 1, 2]
METHODS = ['source', 'tent', 'sar', 'eata', 'rdumb', 'bmia_fixed', 'bmia_adaptive']

In [ ]:
# ================================================================
# Data Loading
# ================================================================

DATA_PATH = None
for root, dirs, files in os.walk('/kaggle/input/'):
    if 'labels.npy' in files and 'gaussian_noise.npy' in files:
        if 'CIFAR-100-C' in root or 'cifar100' in root.lower():
            DATA_PATH = root
            break
assert DATA_PATH is not None, 'CIFAR-100-C not found!'
print(f'CIFAR-100-C at: {DATA_PATH}')

ALL_LABELS = np.load(os.path.join(DATA_PATH, 'labels.npy'))
print(f'Labels: {ALL_LABELS.shape}, range {ALL_LABELS.min()}-{ALL_LABELS.max()}')

CIFAR_MEAN = torch.tensor([0.5071, 0.4867, 0.4408]).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2675, 0.2565, 0.2761]).view(1, 3, 1, 1)


def load_corruption(name, severity=5, n_samples=N_SAMPLES):
    data = np.load(os.path.join(DATA_PATH, f'{name}.npy'))
    start = (severity - 1) * 10000
    end = start + 10000
    imgs = data[start:end][:n_samples]
    lbls = ALL_LABELS[start:end][:n_samples]
    X = torch.from_numpy(imgs.copy()).permute(0, 3, 1, 2).float() / 255.0
    X = (X - CIFAR_MEAN) / CIFAR_STD
    y = torch.from_numpy(lbls.copy()).long()
    return X.to(device), y.to(device)

print('Data loading ready.')

In [ ]:
# ================================================================
# Model Training Function (per seed)
# ================================================================

def train_model(seed):
    """Train ResNet-18 on CIFAR-100 with given seed. Returns state_dict + clean_acc."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
    ])
    trainset = torchvision.datasets.CIFAR100(root='/tmp/cifar100', train=True,
                                              download=True, transform=train_transform)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True,
                                              num_workers=2, generator=torch.Generator().manual_seed(seed))
    
    model = torchvision.models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(512, NUM_CLASSES)
    model = model.to(device)
    
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(50):
        model.train()
        for X, y in trainloader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
        scheduler.step()
        if (epoch + 1) % 10 == 0:
            print(f'  Seed {seed}, Epoch {epoch+1}/50')
    
    # Clean accuracy
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
    ])
    testset = torchvision.datasets.CIFAR100(root='/tmp/cifar100', train=False,
                                             download=True, transform=test_transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in testloader:
            X, y = X.to(device), y.to(device)
            correct += (model(X).argmax(1) == y).sum().item()
            total += y.size(0)
    clean_acc = correct / total
    print(f'  Seed {seed}: clean_acc = {clean_acc:.4f}')
    
    return copy.deepcopy(model.state_dict()), clean_acc, model

In [ ]:
# ================================================================
# TTA Methods (same as run)
# ================================================================

def freeze_except_bn(mdl):
    bn_ids = set()
    for m in mdl.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            for p in m.parameters():
                bn_ids.add(id(p))
    for p in mdl.parameters():
        p.requires_grad_(id(p) in bn_ids)


def get_full_metrics(mdl, X, y):
    mdl.eval()
    all_preds, all_probs = [], []
    with torch.no_grad():
        for i in range(0, X.size(0), BATCH_SIZE):
            logits = mdl(X[i:i+BATCH_SIZE])
            probs = torch.softmax(logits, dim=1)
            all_preds.append(logits.argmax(1))
            all_probs.append(probs)
    preds = torch.cat(all_preds)
    probs = torch.cat(all_probs)
    acc = (preds == y[:len(preds)]).float().mean().item()
    counts = torch.bincount(preds, minlength=NUM_CLASSES).float()
    total = counts.sum()
    dom_pct = counts.max().item() / total.item()
    n_classes = (counts > 0).sum().item()
    h_yx = -(probs * torch.log(probs + 1e-8)).sum(1).mean().item()
    marginal = probs.mean(0)
    h_y = -(marginal * torch.log(marginal + 1e-8)).sum().item()
    mi = h_y - h_yx
    return {
        'acc': round(acc, 4), 'dom_pct': round(dom_pct, 4),
        'n_classes': n_classes, 'h_y': round(h_y, 4),
        'h_yx': round(h_yx, 4), 'mi': round(mi, 4),
    }


def run_method(model, SOURCE_STATE, X, y, method='tent', lr=0.01, anchor=1.0, reset_every=0):
    model.load_state_dict(copy.deepcopy(SOURCE_STATE))
    if method == 'source':
        return get_full_metrics(model, X, y)

    model.train()
    freeze_except_bn(model)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.SGD(params, lr=lr, momentum=0.9)

    init_params = None
    fisher = None
    current_lambda = 0.5 if method == 'bmia_adaptive' else anchor
    batch_count = 0

    if method in ['bmia_fixed', 'bmia_adaptive', 'eata']:
        init_params = {pn: p.data.clone()
                       for pn, p in model.named_parameters() if p.requires_grad}

    if method == 'eata':
        fisher = {pn: torch.zeros_like(p)
                  for pn, p in model.named_parameters() if p.requires_grad}
        n_fisher = min(128, X.size(0))
        for j in range(0, n_fisher, BATCH_SIZE):
            xb = X[j:j+BATCH_SIZE]
            model.zero_grad()
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)
            ent = -(probs * torch.log(probs + 1e-8)).sum(1).mean()
            ent.backward()
            for pn, p in model.named_parameters():
                if pn in fisher and p.grad is not None:
                    fisher[pn] += p.grad.data ** 2 * xb.size(0)
        for pn in fisher:
            fisher[pn] /= n_fisher
        model.load_state_dict(copy.deepcopy(SOURCE_STATE))
        model.train()
        freeze_except_bn(model)
        params = [p for p in model.parameters() if p.requires_grad]
        opt = optim.SGD(params, lr=lr, momentum=0.9)

    for i in range(0, X.size(0), BATCH_SIZE):
        xb = X[i:i+BATCH_SIZE]
        if xb.size(0) < 4:
            continue

        if method == 'rdumb':
            batch_count += 1
            if reset_every > 0 and batch_count % reset_every == 0:
                model.load_state_dict(copy.deepcopy(SOURCE_STATE))
                model.train()
                freeze_except_bn(model)
                params = [p for p in model.parameters() if p.requires_grad]
                opt = optim.SGD(params, lr=lr, momentum=0.9)

        opt.zero_grad()
        logits = model(xb)
        probs = torch.softmax(logits, dim=1)

        if method in ['tent', 'rdumb']:
            loss = -(probs * torch.log(probs + 1e-8)).sum(1).mean()
        elif method == 'sar':
            ent = -(probs * torch.log(probs + 1e-8)).sum(1)
            reliable = ent < 0.4 * np.log(NUM_CLASSES)
            if reliable.sum() < 2:
                continue
            loss = ent[reliable].mean()
        elif method == 'eata':
            ent = -(probs * torch.log(probs + 1e-8)).sum(1)
            reliable = ent < 0.4 * np.log(NUM_CLASSES)
            if reliable.sum() < 2:
                continue
            ent_loss = ent[reliable].mean()
            fisher_loss = sum((fisher[pn] * (p - init_params[pn]) ** 2).sum()
                             for pn, p in model.named_parameters() if pn in fisher)
            loss = ent_loss + 2000 * fisher_loss
        elif method == 'bmia_fixed':
            marginal_ent = -(probs.mean(0) * torch.log(probs.mean(0) + 1e-8)).sum()
            cond_ent = -(probs * torch.log(probs + 1e-8)).sum(1).mean()
            mi_loss = -marginal_ent + cond_ent
            anc_loss = sum(((p - init_params[pn]) ** 2).sum()
                           for pn, p in model.named_parameters() if pn in init_params)
            loss = mi_loss + anchor * anc_loss
        elif method == 'bmia_adaptive':
            marginal_ent = -(probs.mean(0) * torch.log(probs.mean(0) + 1e-8)).sum()
            cond_ent = -(probs * torch.log(probs + 1e-8)).sum(1).mean()
            mi_loss = -marginal_ent + cond_ent
            anc_loss = sum(((p - init_params[pn]) ** 2).sum()
                           for pn, p in model.named_parameters() if pn in init_params)
            loss = mi_loss + current_lambda * anc_loss

        loss.backward()
        opt.step()

        if method == 'bmia_adaptive':
            with torch.no_grad():
                batch_preds = model(xb).argmax(1)
                counts = torch.bincount(batch_preds, minlength=NUM_CLASSES).float()
                dom = counts.max().item() / counts.sum().item()
            if dom > 0.10:
                current_lambda = min(10.0, current_lambda * 1.1)
            else:
                current_lambda = max(0.01, current_lambda * 0.95)

    return get_full_metrics(model, X, y)

print('Methods defined.')

In [ ]:
# ================================================================
# Run Multi-Seed Experiments
# ================================================================

all_seed_results = []

for seed in SEEDS:
    print(f'\n{"="*80}')
    print(f'  SEED: {seed}')
    print(f'{"="*80}')
    
    # Train model with this seed
    SOURCE_STATE, clean_acc, model = train_model(seed)
    
    # Also set numpy/torch seed for TTA randomness
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    for sev in SEVERITIES:
        for corr in CORRUPTIONS:
            X, y = load_corruption(corr, severity=sev, n_samples=N_SAMPLES)
            
            for method in METHODS:
                for lr in LR_LIST:
                    if method == 'source' and lr != LR_LIST[0]:
                        src = [r for r in all_seed_results
                               if r['seed'] == seed and r['severity'] == sev
                               and r['corruption'] == corr and r['method'] == 'source'][0]
                        all_seed_results.append({**src, 'lr': lr})
                        continue
                    
                    t0 = time.time()
                    if method == 'rdumb':
                        r = run_method(model, SOURCE_STATE, X, y, method='rdumb', lr=lr, reset_every=10)
                    elif method == 'bmia_fixed':
                        r = run_method(model, SOURCE_STATE, X, y, method='bmia_fixed', lr=lr, anchor=1.0)
                    else:
                        r = run_method(model, SOURCE_STATE, X, y, method=method, lr=lr)
                    tt = time.time() - t0
                    
                    collapsed = (r['dom_pct'] > 0.15 and r['n_classes'] < 50) or r['n_classes'] < 20
                    
                    print(f'  s{seed} {corr[:8]:8s} {method:16s} lr={lr:.3f}: '
                          f'acc={r["acc"]:.4f} dom={r["dom_pct"]:.1%} '
                          f'col={"Y" if collapsed else "N"} ({tt:.0f}s)')
                    
                    all_seed_results.append({
                        'seed': seed, 'clean_acc': clean_acc,
                        'severity': sev, 'corruption': corr,
                        'method': method, 'lr': lr,
                        **r, 'collapsed': collapsed
                    })
            
            del X, y; gc.collect(); torch.cuda.empty_cache()
    
    del model; gc.collect(); torch.cuda.empty_cache()

print(f'\nDone. Total results: {len(all_seed_results)}')

In [ ]:
# ================================================================
# Results: Mean ± Std Across Seeds
# ================================================================

print('=' * 100)
print('MULTI-SEED RESULTS: Mean ± Std (seeds 0, 1, 2)')
print('Combine with run seed=42 for 4-seed statistics')
print('=' * 100)

for lr in LR_LIST:
    print(f'\n--- lr={lr}, severity=5 ---')
    print(f'{"Method":<18} {"Avg Acc ± Std":>18} {"Collapses":>12} {"Per-seed accs":>40}')
    print('-' * 90)
    
    for method in METHODS:
        seed_avgs = []
        seed_collapses = []
        
        for seed in SEEDS:
            seed_data = [r for r in all_seed_results
                         if r['seed'] == seed and r['method'] == method
                         and r['lr'] == lr and r['severity'] == 5]
            if seed_data:
                avg_acc = np.mean([r['acc'] for r in seed_data]) * 100
                n_col = sum(1 for r in seed_data if r['collapsed'])
                seed_avgs.append(avg_acc)
                seed_collapses.append(n_col)
        
        if seed_avgs:
            mean_acc = np.mean(seed_avgs)
            std_acc = np.std(seed_avgs)
            total_col = sum(seed_collapses)
            total_runs = len(CORRUPTIONS) * len(SEEDS)
            per_seed = ', '.join([f'{a:.1f}' for a in seed_avgs])
            print(f'{method:<18} {mean_acc:>8.1f} ± {std_acc:<7.1f} '
                  f'{total_col:>4d}/{total_runs:<6d} [{per_seed}]')

# Clean accuracy across seeds
print(f'\n--- Clean Accuracy Across Seeds ---')
for seed in SEEDS:
    seed_data = [r for r in all_seed_results if r['seed'] == seed]
    if seed_data:
        print(f'  Seed {seed}: clean_acc = {seed_data[0]["clean_acc"]:.4f}')

In [ ]:
# ================================================================
# Per-Corruption Breakdown
# ================================================================

print('\n' + '=' * 100)
print('PER-CORRUPTION BREAKDOWN (severity=5)')
print('=' * 100)

for lr in LR_LIST:
    print(f'\n--- lr={lr} ---')
    for method in ['tent', 'eata', 'bmia_adaptive']:
        print(f'\n  {method}:')
        for corr in CORRUPTIONS:
            accs = []
            cols = []
            for seed in SEEDS:
                r = [x for x in all_seed_results
                     if x['seed'] == seed and x['method'] == method
                     and x['lr'] == lr and x['severity'] == 5
                     and x['corruption'] == corr]
                if r:
                    accs.append(r[0]['acc'] * 100)
                    cols.append(r[0]['collapsed'])
            if accs:
                print(f'    {corr:<20} {np.mean(accs):>6.1f} ± {np.std(accs):<5.1f} '
                      f'collapsed: {sum(cols)}/{len(cols)}')

In [ ]:
# ================================================================
# Collapse Rate Summary
# ================================================================

print('\n' + '=' * 100)
print('COLLAPSE RATE SUMMARY (severity=5, all seeds)')
print('=' * 100)

for method in METHODS:
    if method == 'source':
        continue
    m_data = [r for r in all_seed_results
              if r['method'] == method and r['severity'] == 5]
    n_col = sum(1 for r in m_data if r['collapsed'])
    total = len(m_data)
    if total > 0:
        print(f'  {method:<18}: {n_col}/{total} ({n_col/total:.1%})')

print('\n*** KEY QUESTION: Does BMIA maintain 0 collapses across all seeds? ***')
bmia_data = [r for r in all_seed_results
             if r['method'] in ['bmia_fixed', 'bmia_adaptive']]
bmia_col = sum(1 for r in bmia_data if r['collapsed'])
print(f'  BMIA total: {bmia_col}/{len(bmia_data)} collapses')
print(f'  Answer: {"YES — 0 collapses across all seeds" if bmia_col == 0 else "NO — collapses detected!"}')

In [ ]:
# Save
save_data = {
    'experiment': 'V12_MULTISEED',
    'seeds': SEEDS,
    'note': 'Combine with run seed=42 for 4-seed statistics',
    'results': all_seed_results,
}
with open('V12_MULTISEED_results.json', 'w') as f:
    json.dump(save_data, f, indent=2, default=str)
print('Saved to V12_MULTISEED_results.json')
print('\n*** SHARE THIS FULL OUTPUT WITH THE PANEL ***')